# SHIPP Stage 16 — Security Validation

**Owner:** Raneem  
**Priority:** P0  
**Purpose:** Read-only security validation for SHIPP.

This notebook validates the P0 security boundary without changing working pipeline, Gold, Agent, or Lakebase state.

## Gates

- **16.1 Repository secret hygiene** — scan repository source files for configured high-risk credential patterns without printing secret values.
- **16.2 Downstream privacy / data minimization** — verify Silver, Gold, and Search schemas do not expose prohibited fields.
- **16.3 Searchable-content PII check** — validate that searchable listing content does not contain email or UAE-mobile patterns.
- **16.4 Agent write audit controls** — reuse existing Lakebase CDC audit evidence; this notebook performs no business write.

**Read-only rule:** this notebook must not call `save_item()`, modify Lakebase, rewrite Gold, or print credentials.


In [0]:
# ============================================================
# STAGE 16 — GATE 16.1
# REPOSITORY SECRET HYGIENE
# READ ONLY
# ============================================================

import os
import re
from pathlib import Path

print("=== STAGE 16.1 — REPOSITORY SECRET HYGIENE ===")


def find_repo_root(start: str) -> Path:
    path = Path(start).resolve()

    while True:
        if (
            (path / "config").is_dir()
            and (path / "data_pipeline").is_dir()
        ):
            return path

        if path.parent == path:
            raise RuntimeError(
                f"Could not locate SHIPP repo root above {start}"
            )

        path = path.parent


REPO_ROOT = find_repo_root(os.getcwd())
SELF_PATH = Path("notebooks/validation/16_validate_security.ipynb")

print("Repo root:", REPO_ROOT)


SKIP_DIRS = {
    ".git",
    ".venv",
    "venv",
    "__pycache__",
    ".pytest_cache",
}

TEXT_SUFFIXES = {
    ".py",
    ".sql",
    ".md",
    ".yml",
    ".yaml",
    ".json",
    ".ipynb",
    ".toml",
    ".txt",
}

# Intentionally narrow, high-confidence patterns.
# Never print the matched credential text itself.
SECRET_RULES = {
    "hardcoded Databricks PAT": re.compile(
        r"dapi[A-Za-z0-9]{20,}"
    ),
    "hardcoded ORS secret write": re.compile(
        r"(?is)put_secret\s*\([\s\S]{0,1500}?"
        r"ors-api-key[\s\S]{0,1500}?"
        r"string_value\s*=\s*[\"'][^\"']{12,}[\"']"
    ),
}


violations = []

for path in REPO_ROOT.rglob("*"):

    if not path.is_file():
        continue

    relative = path.relative_to(REPO_ROOT)

    # Avoid this validator matching the regex examples embedded in itself.
    if relative == SELF_PATH:
        continue

    if any(part in SKIP_DIRS for part in relative.parts):
        continue

    if path.suffix.lower() not in TEXT_SUFFIXES:
        continue

    try:
        content = path.read_text(
            encoding="utf-8",
            errors="ignore",
        )
    except Exception:
        continue

    for rule_name, pattern in SECRET_RULES.items():
        if pattern.search(content):
            violations.append(
                {
                    "file": str(relative),
                    "rule": rule_name,
                }
            )


print("Potential secret violations:", len(violations))

for violation in violations:
    print(
        " -",
        violation["file"],
        "→",
        violation["rule"],
    )


assert not violations, (
    "FAILED — possible hardcoded credential material "
    f"exists in repository files: {violations}"
)

print(
    "PASS Gate 16.1 — no configured hardcoded-secret "
    "patterns found in repository source."
)


In [0]:
# ============================================================
# STAGE 16 — GATE 16.2
# DOWNSTREAM PRIVACY / DATA MINIMIZATION
# READ ONLY
# ============================================================

print("=== STAGE 16.2 — DOWNSTREAM PRIVACY BOUNDARY ===")


COMMON_FORBIDDEN = {
    "email",
    "email_address",
    "phone",
    "phone_number",
    "password",
    "password_hash",
    "api_key",
    "secret",
    "access_token",
    "refresh_token",
}


TABLE_RULES = {
    "bootcamp_students.shipp_silver.silver_listings": {
        "forbidden_columns": COMMON_FORBIDDEN,
    },

    "bootcamp_students.shipp_silver.silver_requests": {
        "forbidden_columns": COMMON_FORBIDDEN,
    },

    # Gold candidate matching is structured serving data.
    # Raw user-entered text and exact coordinates are not required by
    # the Agent candidate contract.
    "bootcamp_students.shipp_gold.gold_candidate_matches": {
        "forbidden_columns": COMMON_FORBIDDEN
        | {
            "request_text",
            "listing_description",
            "latitude",
            "longitude",
            "request_latitude",
            "request_longitude",
            "listing_latitude",
            "listing_longitude",
        },
    },

    # AI Search may contain cleaned searchable text, but not direct
    # contact fields or exact coordinates.
    "bootcamp_students.shipp_gold.gold_listing_search_docs": {
        "forbidden_columns": COMMON_FORBIDDEN
        | {
            "latitude",
            "longitude",
            "request_latitude",
            "request_longitude",
            "listing_latitude",
            "listing_longitude",
        },
    },
}


violations = []
checked_tables = []


for table_name, rules in TABLE_RULES.items():

    print()
    print("----------------------------------------")
    print("TABLE:", table_name)

    if not spark.catalog.tableExists(table_name):
        print("SKIP — table does not exist")
        continue

    checked_tables.append(table_name)

    actual_columns = {
        column.lower()
        for column in spark.table(table_name).columns
    }

    forbidden = {
        column.lower()
        for column in rules["forbidden_columns"]
    }

    found = sorted(
        actual_columns.intersection(forbidden)
    )

    print("Forbidden columns found:", found)

    if found:
        violations.append(
            {
                "table": table_name,
                "columns": found,
            }
        )


assert checked_tables, (
    "BLOCKED — none of the expected downstream "
    "SHIPP tables were available."
)

assert not violations, (
    "FAILED — privacy/data-minimization violations "
    f"found downstream: {violations}"
)

print()
print("Checked tables:", len(checked_tables))
print(
    "PASS Gate 16.2 — downstream Silver/Gold/Search "
    "schemas satisfy the SHIPP data-minimization boundary."
)


In [0]:
# ============================================================
# STAGE 16 — GATE 16.3
# SEARCHABLE CONTENT PII CHECK
# READ ONLY
# ============================================================

from pyspark.sql import functions as F

print("=== STAGE 16.3 — SEARCHABLE CONTENT PII CHECK ===")

SEARCH_DOCS = (
    "bootcamp_students.shipp_gold."
    "gold_listing_search_docs"
)

assert spark.catalog.tableExists(SEARCH_DOCS), (
    f"BLOCKED — search docs table does not exist: "
    f"{SEARCH_DOCS}"
)

search_docs = spark.table(SEARCH_DOCS)

text_columns = [
    column
    for column in [
        "title",
        "description",
        "image_summary",
        "visual_attributes",
        "search_text",
    ]
    if column in search_docs.columns
]

assert text_columns, (
    "BLOCKED — no searchable text columns found."
)

combined_text = F.concat_ws(
    " ",
    *[
        F.coalesce(
            F.col(column).cast("string"),
            F.lit(""),
        )
        for column in text_columns
    ],
)

EMAIL_PATTERN = (
    r"(?i)\b[A-Z0-9._%+-]+@"
    r"[A-Z0-9.-]+\.[A-Z]{2,}\b"
)

# UAE mobile forms such as 0501234567, +971501234567, or 00971501234567.
# This deliberately avoids generic long-number matching so dimensions/dates
# are not treated as phone numbers.
PHONE_PATTERN = (
    r"(?i)(?:\+971|00971|0)5\d"
    r"(?:[\s().-]*\d){7}"
)

email_hits = (
    search_docs
    .filter(combined_text.rlike(EMAIL_PATTERN))
    .count()
)

phone_hits = (
    search_docs
    .filter(combined_text.rlike(PHONE_PATTERN))
    .count()
)

print("Search documents:", search_docs.count())
print("Email-like values found:", email_hits)
print("UAE-mobile-like values found:", phone_hits)

assert email_hits == 0, (
    f"FAILED — {email_hits} search document(s) "
    "contain email-like text."
)

assert phone_hits == 0, (
    f"FAILED — {phone_hits} search document(s) "
    "contain UAE-mobile-like text."
)

print(
    "PASS Gate 16.3 — no configured email/phone "
    "patterns found in searchable listing content."
)


In [0]:
# ============================================================
# STAGE 16 — GATE 16.4
# AGENT WRITE AUDIT CONTROLS
# READ ONLY — REUSE EXISTING STAGE 12 EVIDENCE
# ============================================================

from pyspark.sql import functions as F

print("=== STAGE 16.4 — AGENT WRITE AUDIT CONTROLS ===")

AGENT_ACTIVITY_HISTORY = (
    "bootcamp_students.shipp_bronze."
    "lb_agent_activity_history"
)

assert spark.catalog.tableExists(AGENT_ACTIVITY_HISTORY), (
    f"BLOCKED — missing Agent activity CDF table: "
    f"{AGENT_ACTIVITY_HISTORY}"
)

activity = spark.table(AGENT_ACTIVITY_HISTORY)

required_columns = {
    "activity_id",
    "tool_name",
    "action_status",
    "_pg_change_type",
    "_pg_lsn",
    "_sort_by",
}

missing_columns = sorted(
    required_columns - set(activity.columns)
)

assert not missing_columns, (
    "FAILED — Agent activity history is missing "
    f"required columns: {missing_columns}"
)

# Agent activity is an append-only audit record in Lakebase. Restrict to
# current-value event forms in case the source later emits updates.
save_activity = (
    activity
    .filter(
        F.lower(F.col("_pg_change_type"))
        .isin("insert", "update_postimage")
    )
    .filter(
        F.lower(F.col("tool_name")) == "save_item"
    )
)

save_event_count = save_activity.count()

print("save_item audit events:", save_event_count)

assert save_event_count > 0, (
    "BLOCKED — no real save_item audit event exists yet."
)

status_rows = (
    save_activity
    .select(
        F.upper(F.col("action_status"))
        .alias("action_status")
    )
    .distinct()
    .collect()
)

statuses = {
    row["action_status"]
    for row in status_rows
    if row["action_status"] is not None
}

print("Observed save_item statuses:", sorted(statuses))

required_statuses = {
    "SUCCESS",
    "REJECTED_DUPLICATE",
    "REJECTED_UNAVAILABLE",
}

missing_statuses = sorted(
    required_statuses - statuses
)

assert not missing_statuses, (
    "BLOCKED — Stage 12 security evidence is incomplete. "
    f"Missing save_item audit status(es): {missing_statuses}"
)

display(
    save_activity
    .select(
        "activity_id",
        "tool_name",
        "action_status",
        "_pg_change_type",
        "_pg_lsn",
        "_sort_by",
    )
    .orderBy(
        F.col("_pg_lsn").desc(),
        F.col("_sort_by").desc(),
    )
    .limit(20)
)

print(
    "PASS Gate 16.4 — existing Agent write audit evidence "
    "proves success, duplicate rejection, and unavailable rejection."
)


In [0]:
print("==============================================")
print("SHIPP STAGE 16 — SECURITY VALIDATION")
print("==============================================")
print("PASS 16.1 — repository secret hygiene")
print("PASS 16.2 — downstream data minimization")
print("PASS 16.3 — searchable-content PII validation")
print("PASS 16.4 — Agent write audit controls")
print()
print("PASS — STAGE 16 SECURITY VALIDATED")
